# SyNG-BTS Evaluation Template

Evaluates one synthetic dataset against real/empirical data.

**Sections:**
- **A** — Feature Structure: Summary Statistic Preservation (mean, SD, median, IQR, skewness, kurtosis, sparsity + MAD)
- **B** — Feature Structure: Marginal Distributions (Wasserstein-1, KS, matched-size bootstrap)
- **C** — Sample Structure: Joint UMAP Embedding
- **D** — Sample Structure: Mixing Metrics (kNN mixing k=10, silhouette, cARI, bootstrap)
- **E.1** — Correlation Structure: ΔR Heatmap
- **E.2** — Correlation Structure: Scalar Metrics (median |Δρ|, Frobenius norm, Mantel r, bootstrap)
- **E.3** — Correlation Structure: Pearson Threshold Analysis (±0.3 class agreement, false-correlation rate)
- **E.4** — Correlation Structure: Discretized Joint Distributions (bivariate accuracy per pair)

**Setup required:** Edit the `BASE_DIR`, `REAL_FILE`, `GEN_FILE`, and `GENERATOR_LABEL` variables in Chunk 2 before running.

In [ ]:
# Chunk 1 — Setup: imports and display defaults
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import wasserstein_distance, ks_2samp
from sklearn.preprocessing import StandardScaler, KBinsDiscretizer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.cluster import KMeans

try:
    import umap  # type: ignore
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("umap-learn not installed — Section C will be skipped. Install with: pip install umap-learn")

sns.set(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [ ]:
# Chunk 2 — Config: edit these paths and parameters before running

# ── File paths ──────────────────────────────────────────────────────────────
BASE_DIR        = Path(".")                       # directory containing your CSV files
REAL_FILE       = "your_real_data.csv"            # empirical / training data
GEN_FILE        = "your_generated_data.csv"       # synthetic output (may lack a header row)
GENERATOR_LABEL = "Model"                         # display label used in plot titles, e.g. "CVAE1-1"

# ── Bootstrap ───────────────────────────────────────────────────────────────
N_BOOTSTRAP           = 300
BOOTSTRAP_RANDOM_SEED = 42

# ── Mixing metrics ───────────────────────────────────────────────────────────
K_NEIGHBORS = 10          # k for kNN mixing score (per-specification: k=10)

# ── Correlation thresholds (Section E.3) ────────────────────────────────────
CORR_THRESH_POS = 0.3     # pairs with r > threshold → "strong positive"
CORR_THRESH_NEG = -0.3    # pairs with r < threshold → "strong negative"

# ── Discretized joint distributions (Section E.4) ───────────────────────────
MAX_BINS  = 10     # max equal-frequency bins per feature, fit on real data
MAX_PAIRS = None   # None = all C(D,2) pairs; set e.g. 5000 to limit computation

# ── Columns to exclude from feature analysis ─────────────────────────────────
LABEL_CANDIDATES = ["groups", "groups2"]

print(f"Real file : {BASE_DIR / REAL_FILE}")
print(f"Gen file  : {BASE_DIR / GEN_FILE}")
print(f"Generator : {GENERATOR_LABEL}")
print(f"Bootstrap : N={N_BOOTSTRAP}, seed={BOOTSTRAP_RANDOM_SEED}")

In [ ]:
# Chunk 3 — Data loading: load CSVs, align feature columns, median-impute NaNs

def _to_numeric_features(df: pd.DataFrame) -> pd.DataFrame:
    drop = [c for c in LABEL_CANDIDATES if c in df.columns]
    out = df.drop(columns=drop, errors="ignore").select_dtypes(include=[np.number]).copy()
    return out.replace([np.inf, -np.inf], np.nan)


def load_and_align(real_path: Path, gen_path: Path):
    real_df = pd.read_csv(real_path)
    gen_raw = pd.read_csv(gen_path, header=None)

    real_numeric_cols = [c for c in real_df.columns if c not in LABEL_CANDIDATES]
    if gen_raw.shape[1] == real_df.shape[1]:
        gen_raw.columns = real_df.columns.tolist()
    elif gen_raw.shape[1] == len(real_numeric_cols):
        gen_raw.columns = real_numeric_cols
    else:
        raise ValueError(
            f"Column mismatch: generated has {gen_raw.shape[1]} columns, "
            f"real has {real_df.shape[1]} total / {len(real_numeric_cols)} numeric."
        )

    real_x = _to_numeric_features(real_df)
    gen_x  = _to_numeric_features(gen_raw)

    common = [c for c in real_x.columns if c in gen_x.columns]
    if not common:
        raise ValueError("No common numeric features between real and generated data.")

    real_x, gen_x = real_x[common].copy(), gen_x[common].copy()

    for df in (real_x, gen_x):
        nan_cols = df.columns[df.isna().any()].tolist()
        for col in nan_cols:
            df[col] = df[col].fillna(df[col].median())

    return real_x, gen_x


real_x, gen_x = load_and_align(BASE_DIR / REAL_FILE, BASE_DIR / GEN_FILE)
FEATURES = real_x.columns.tolist()

print(f"Real shape : {real_x.shape}")
print(f"Gen shape  : {gen_x.shape}")
print(f"Features   : {len(FEATURES)}")

---
## A. Feature Structure: Summary Statistic Preservation

Per-feature summary statistics are computed for real and synthetic data. Preservation is quantified using the
Median Absolute Deviation (MAD) of the absolute differences across features; values near zero indicate high fidelity.

Statistics: **mean, SD, median, IQR, skewness, kurtosis, sparsity** (% zeros).

In [ ]:
# Chunk 4 — A: Summary statistics per feature + MAD aggregation

def _sparsity(s: pd.Series) -> float:
    return float((s == 0).mean() * 100)

SUMMARY_STATS = {
    "mean":     lambda s: float(s.mean()),
    "sd":       lambda s: float(s.std(ddof=1)),
    "median":   lambda s: float(s.median()),
    "iqr":      lambda s: float(s.quantile(0.75) - s.quantile(0.25)),
    "skewness": lambda s: float(s.skew()),
    "kurtosis": lambda s: float(s.kurt()),
    "sparsity": _sparsity,
}


def _build_stat_table(real_x: pd.DataFrame, gen_x: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for feat in real_x.columns:
        for stat_name, fn in SUMMARY_STATS.items():
            rv = fn(real_x[feat])
            gv = fn(gen_x[feat])
            rows.append({"feature": feat, "statistic": stat_name,
                         "real_value": rv, "gen_value": gv, "abs_diff": abs(rv - gv)})
    return pd.DataFrame(rows)


def _mad(values) -> float:
    v = np.asarray(values, float)
    return float(np.median(np.abs(v - np.median(v))))


stat_table = _build_stat_table(real_x, gen_x)

# MAD of |Δ statistic| across features
mad_series = (
    stat_table.groupby("statistic")["abs_diff"]
    .apply(lambda x: _mad(x.to_numpy()))
    .reindex(list(SUMMARY_STATS.keys()))
)
mad_df = mad_series.to_frame(name=GENERATOR_LABEL).T
mad_df.index.name = "generator"

print("MAD of |Δ statistic| across features:")
display(mad_df)

# Heatmap
fig, ax = plt.subplots(figsize=(11, 2.4))
sns.heatmap(mad_df, annot=True, fmt=".4g", cmap="YlOrRd", ax=ax)
ax.set_title(f"A. Summary Statistic MAD — {GENERATOR_LABEL}")
ax.set_xlabel("Statistic")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

# Distribution of per-feature |Δ| for each statistic
g = sns.FacetGrid(
    stat_table,
    col="statistic", col_wrap=4,
    col_order=list(SUMMARY_STATS.keys()),
    height=2.8, aspect=1.1,
    sharey=False,
)
g.map(sns.histplot, "abs_diff", bins=30, color="steelblue")
g.set_axis_labels("|Δ|", "count")
g.set_titles("{col_name}")
g.figure.suptitle(f"Per-feature |Δ statistic| distributions — {GENERATOR_LABEL}", y=1.02)
plt.tight_layout()
plt.show()

---
## B. Feature Structure: Marginal Distribution Metrics

Each bootstrap replicate draws matched-size subsamples (`min(n_real, n_gen)`) from real and synthetic data.
Within each replicate, **Wasserstein-1 (W₁)** and **KS** statistics are computed per feature and summarized
by the median across features. The resulting bootstrap distributions are shown as violin plots.

- **W₁** (Earth Mover's Distance): minimum transport cost to align two distributions — sensitive to location, scale, and tails.
- **KS**: maximum gap between the two empirical CDFs — captures shape and support differences.

Lower values indicate better marginal fidelity.

In [ ]:
# Chunk 5 — B: Matched-size bootstrap → median W1 / median KS per replicate

def bootstrap_marginals(
    real_x: pd.DataFrame,
    gen_x: pd.DataFrame,
    n_bootstrap: int = N_BOOTSTRAP,
    seed: int = BOOTSTRAP_RANDOM_SEED,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    real_arr = real_x.to_numpy(float)
    gen_arr  = gen_x.to_numpy(float)
    n_real, n_gen = real_arr.shape[0], gen_arr.shape[0]
    n_sample = min(n_real, n_gen)

    records = []
    for b in range(n_bootstrap):
        ir = rng.choice(n_real, n_sample, replace=True)
        ig = rng.choice(n_gen,  n_sample, replace=True)
        r_b, g_b = real_arr[ir], gen_arr[ig]

        w1_vals, ks_vals = [], []
        for j in range(r_b.shape[1]):
            w1_vals.append(float(wasserstein_distance(r_b[:, j], g_b[:, j])))
            ks_vals.append(float(ks_2samp(r_b[:, j], g_b[:, j], method="asymp").statistic))

        records.append({
            "bootstrap_id": b,
            "W1_median":    float(np.median(w1_vals)),
            "KS_median":    float(np.median(ks_vals)),
        })
    return pd.DataFrame(records)


marg_df = bootstrap_marginals(real_x, gen_x)

print("B. Marginal bootstrap summary (median over features per replicate):")
display(marg_df[["W1_median", "KS_median"]].agg(["median", "mean", "std"]))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, col, label in zip(
    axes,
    ["W1_median", "KS_median"],
    ["Median W\u2081 across features", "Median KS across features"],
):
    sns.violinplot(y=marg_df[col], inner="box", cut=0, ax=ax, color="steelblue")
    ax.set_title(f"{label}\n{GENERATOR_LABEL}")
    ax.set_ylabel(label)
    ax.set_xticks([])
fig.suptitle("B. Marginal Distribution Metrics (matched-size bootstrap)")
plt.tight_layout()
plt.show()

---
## C. Sample Structure: Joint UMAP Embedding

Real and synthetic samples are jointly embedded in 2D using UMAP after standard scaling.
Visual overlap of the two point clouds indicates that synthetic samples occupy the same data manifold as real samples.

**Requires:** `pip install umap-learn`

In [ ]:
# Chunk 6 — C: Joint UMAP embedding of real + synthetic

if HAS_UMAP:
    X_joint = np.vstack([
        real_x[FEATURES].to_numpy(float),
        gen_x[FEATURES].to_numpy(float),
    ])
    n_real_umap = len(real_x)

    Xs_joint = StandardScaler().fit_transform(X_joint)
    n_nb = min(15, max(2, Xs_joint.shape[0] - 1))
    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=n_nb, min_dist=0.1)
    Z = reducer.fit_transform(Xs_joint)
    z_real_umap = Z[:n_real_umap]
    z_gen_umap  = Z[n_real_umap:]

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(z_real_umap[:, 0], z_real_umap[:, 1],
               alpha=0.5, s=14, label="real", rasterized=True)
    ax.scatter(z_gen_umap[:, 0],  z_gen_umap[:, 1],
               alpha=0.5, s=14, label=GENERATOR_LABEL, rasterized=True)
    ax.legend()
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.set_title(f"C. UMAP — real vs {GENERATOR_LABEL}")
    plt.tight_layout()
    plt.show()
else:
    print("UMAP not available. Install with: pip install umap-learn")

---
## D. Sample Structure: Mixing Metrics

Three complementary metrics assess how indistinguishable real and synthetic samples are in feature space.
Each is computed on matched-size bootstrap replicates.

| Metric | Scale | Good mixing |
|---|---|---|
| **kNN Mixing** (k=10) | 0–1 | near **1** — local neighborhoods reflect global mix |
| **Silhouette** (real vs gen) | −1–1 | near **0 or negative** — weak source separability |
| **cARI** (1 − |ARI|) | 0–1 | near **1** — k-means clusters don't align with source labels |

**kNN Mixing:** `1 − mean_i(|p_local(i) − p_global|)` where `p_local` is the fraction of real samples among the k nearest neighbors of sample i and `p_global` is the overall real fraction in the pooled dataset.

**cARI (complementary ARI):** k-means (k=2) is applied to the joint feature space; `cARI = 1 − |ARI(cluster_labels, source_labels)|`. High cARI means the unsupervised partition does not align with the real/synthetic split.

In [ ]:
# Chunk 7 — D: kNN mixing, silhouette, cARI — matched-size bootstrap

def _knn_mixing(X: np.ndarray, y_is_real: np.ndarray, k: int) -> float:
    k_use = max(2, min(k, X.shape[0] - 1))
    p_global = float(np.mean(y_is_real))
    nn = NearestNeighbors(n_neighbors=k_use + 1, algorithm="auto").fit(X)
    _, inds = nn.kneighbors(X)
    p_local = np.mean(y_is_real[inds[:, 1:k_use + 1]], axis=1)
    return float(1.0 - np.mean(np.abs(p_local - p_global)))


def _cari(X: np.ndarray, y_source: np.ndarray, rs: int = 0) -> float:
    """cARI = 1 - |ARI|; high = k-means doesn't split real vs synthetic."""
    km = KMeans(n_clusters=2, random_state=rs, n_init=10)
    return float(1.0 - abs(adjusted_rand_score(y_source, km.fit_predict(X))))


def bootstrap_mixing(
    real_x: pd.DataFrame,
    gen_x: pd.DataFrame,
    n_bootstrap: int = N_BOOTSTRAP,
    seed: int = BOOTSTRAP_RANDOM_SEED,
    k: int = K_NEIGHBORS,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    real_arr = real_x.to_numpy(float)
    gen_arr  = gen_x.to_numpy(float)
    n_real, n_gen = real_arr.shape[0], gen_arr.shape[0]
    n_sample = min(n_real, n_gen)

    records = []
    for b in range(n_bootstrap):
        ir = rng.choice(n_real, n_sample, replace=True)
        ig = rng.choice(n_gen,  n_sample, replace=True)
        r_b, g_b = real_arr[ir], gen_arr[ig]

        X = np.vstack([r_b, g_b])
        y = np.array([1] * n_sample + [0] * n_sample, dtype=np.int32)
        Xs = StandardScaler().fit_transform(X)

        mix = _knn_mixing(Xs, y, k=k)

        try:
            sil = float(silhouette_score(
                Xs, y, metric="euclidean",
                sample_size=min(2000, len(Xs)),
                random_state=b,
            ))
        except Exception:
            sil = float("nan")

        try:
            cari = _cari(Xs, y, rs=b)
        except Exception:
            cari = float("nan")

        records.append({"bootstrap_id": b,
                        "knn_mixing": mix,
                        "silhouette": sil,
                        "cARI": cari})
    return pd.DataFrame(records)


mixing_df = bootstrap_mixing(real_x, gen_x)

print("D. Mixing metric bootstrap summary (median / mean / std):")
display(mixing_df[["knn_mixing", "silhouette", "cARI"]].agg(["median", "mean", "std"]))

metric_meta = [
    ("knn_mixing",  "kNN Mixing Score\n(higher = better, k=10)"),
    ("silhouette",  "Silhouette (real vs gen)\n(lower / negative = better)"),
    ("cARI",        "cARI = 1 \u2212 |ARI|\n(higher = better)"),
]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, (col, label) in zip(axes, metric_meta):
    sns.violinplot(y=mixing_df[col], inner="box", cut=0, ax=ax)
    ax.set_title(f"{label}\n{GENERATOR_LABEL}", fontsize=10)
    ax.set_ylabel(col)
    ax.set_xticks([])
fig.suptitle(f"D. Sample Structure Mixing Metrics — {GENERATOR_LABEL} (bootstrap n={N_BOOTSTRAP})")
plt.tight_layout()
plt.show()

---
## E. Correlation Structure

Four complementary views of pairwise Pearson correlation fidelity.

- **E.1** — Visual ΔR heatmap (full sample)
- **E.2** — Scalar summaries: median |Δρ|, Frobenius ‖ΔR‖_F, Mantel r (bootstrap)
- **E.3** — Threshold-based pair classification (±0.3): class agreement rate and false-correlation rate
- **E.4** — Discretized joint distributions: bivariate accuracy per pair (TV distance in binned space)

In [ ]:
# Chunk 8 — Correlation matrices (full-sample; reused by E.1, E.3, E.4)

def _corr_matrix(X: np.ndarray) -> np.ndarray:
    C = np.corrcoef(X, rowvar=False)
    return np.nan_to_num(C, nan=0.0, posinf=0.0, neginf=0.0)


Cr = _corr_matrix(real_x[FEATURES].to_numpy(float))   # shape (D, D)
Cg = _corr_matrix(gen_x[FEATURES].to_numpy(float))
dR = Cr - Cg

print(f"Correlation matrices computed: shape {Cr.shape}")

In [ ]:
# Chunk 9 — E.1: ΔR heatmap (R_real − R_generated)

v = max(float(np.quantile(np.abs(dR), 0.99)), 1e-6)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    pd.DataFrame(dR, index=FEATURES, columns=FEATURES),
    cmap="coolwarm", center=0.0, vmin=-v, vmax=v,
    ax=ax, square=True, xticklabels=False, yticklabels=False,
    cbar_kws={"shrink": 0.75},
)
ax.set_title(f"E.1 Correlation Difference \u0394R = R_real \u2212 R_{GENERATOR_LABEL}")
ax.set_xlabel("feature")
ax.set_ylabel("feature")
plt.tight_layout()
plt.show()

In [ ]:
# Chunk 10 — E.2: Correlation scalar metrics — matched-size bootstrap
# Metrics: median |Δρ| (off-diagonal), Frobenius ||ΔR||_F, Mantel r
# Lower median|Δρ| and Frobenius → better; higher Mantel → better

def _corr_scalars(Cr: np.ndarray, Cg: np.ndarray):
    dR_b = Cr - Cg
    p    = Cr.shape[0]
    off  = ~np.eye(p, dtype=bool)
    med_abs = float(np.median(np.abs(dR_b[off])))
    frob    = float(np.linalg.norm(dR_b, "fro"))
    iu = np.triu_indices(p, k=1)
    vr, vg = Cr[iu], Cg[iu]
    mantel  = float(np.corrcoef(vr, vg)[0, 1]) if np.std(vr) > 1e-12 else float("nan")
    return med_abs, frob, mantel


def bootstrap_corr_scalars(
    real_x: pd.DataFrame,
    gen_x: pd.DataFrame,
    n_bootstrap: int = N_BOOTSTRAP,
    seed: int = BOOTSTRAP_RANDOM_SEED,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    real_arr = real_x.to_numpy(float)
    gen_arr  = gen_x.to_numpy(float)
    n_real, n_gen = real_arr.shape[0], gen_arr.shape[0]
    n_sample = min(n_real, n_gen)

    records = []
    for b in range(n_bootstrap):
        ir = rng.choice(n_real, n_sample, replace=True)
        ig = rng.choice(n_gen,  n_sample, replace=True)
        Cr_b = _corr_matrix(real_arr[ir])
        Cg_b = _corr_matrix(gen_arr[ig])
        med_abs, frob, mantel = _corr_scalars(Cr_b, Cg_b)
        records.append({"bootstrap_id": b,
                        "median_abs_delta_rho": med_abs,
                        "frobenius_norm":       frob,
                        "mantel_r":             mantel})
    return pd.DataFrame(records)


corr_scalar_df = bootstrap_corr_scalars(real_x, gen_x)

print("E.2 Correlation scalar bootstrap summary:")
display(corr_scalar_df[["median_abs_delta_rho", "frobenius_norm", "mantel_r"]].agg(["median", "mean", "std"]))

scalar_meta = [
    ("median_abs_delta_rho", "Median |\u0394\u03c1|\n(lower = better)"),
    ("frobenius_norm",       "Frobenius \u2016\u0394R\u2016_F\n(lower = better)"),
    ("mantel_r",             "Mantel r\n(higher = better)"),
]
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, (col, label) in zip(axes, scalar_meta):
    sns.violinplot(y=corr_scalar_df[col], inner="box", cut=0, ax=ax)
    ax.set_title(f"{label}\n{GENERATOR_LABEL}", fontsize=10)
    ax.set_ylabel(col)
    ax.set_xticks([])
fig.suptitle(f"E.2 Correlation Scalar Metrics — {GENERATOR_LABEL} (bootstrap n={N_BOOTSTRAP})")
plt.tight_layout()
plt.show()

In [ ]:
# Chunk 11 — E.3: Pearson threshold analysis (full-sample)
#
# Every unordered feature pair is labeled using real-data Pearson r:
#   positive : r > +CORR_THRESH_POS
#   negative : r < CORR_THRESH_NEG
#   neutral  : |r| <= threshold
#
# Metrics:
#   Agreement rate     — fraction of pairs with the same label in real and synthetic
#   Stratum agreement  — agreement rate computed separately for each real-data stratum
#   False-correlation  — fraction of real-neutral pairs that become correlated in synthetic

def _label_r(r: float) -> str:
    if r > CORR_THRESH_POS:
        return "positive"
    if r < CORR_THRESH_NEG:
        return "negative"
    return "neutral"


D = len(FEATURES)
iu = np.triu_indices(D, k=1)
r_real_pairs = Cr[iu]
r_gen_pairs  = Cg[iu]

labels_real = np.array([_label_r(r) for r in r_real_pairs])
labels_gen  = np.array([_label_r(r) for r in r_gen_pairs])

# Global agreement
agree = labels_real == labels_gen
global_agr = float(agree.mean())

# Stratum-wise agreement
strata_agr = {}
for st in ["positive", "negative", "neutral"]:
    mask = labels_real == st
    strata_agr[st] = float(agree[mask].mean()) if mask.sum() > 0 else float("nan")

# False-correlation rate
neutral_mask = labels_real == "neutral"
false_corr = float((labels_gen[neutral_mask] != "neutral").mean()) if neutral_mask.sum() > 0 else float("nan")

e3_summary = pd.DataFrame(
    {
        "global_agreement":    [global_agr],
        "agreement_positive":  [strata_agr.get("positive", float("nan"))],
        "agreement_negative":  [strata_agr.get("negative", float("nan"))],
        "agreement_neutral":   [strata_agr.get("neutral",  float("nan"))],
        "false_correlation_rate": [false_corr],
    },
    index=[GENERATOR_LABEL],
)

print(f"E.3 Pearson Threshold Analysis (thresholds \u00b1{CORR_THRESH_POS}):")
print(f"  Total pairs : {len(r_real_pairs):,}")
for st in ["positive", "negative", "neutral"]:
    n_st = int((labels_real == st).sum())
    print(f"  {st:8s} : {n_st:,} ({100 * n_st / len(labels_real):.1f}%)")
print()
display(e3_summary)

# Bar charts
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bar_labels = ["Global", "Positive\n(r>0.3)", "Negative\n(r<\u22120.3)", "Neutral\n(|r|\u22640.3)"]
bar_vals = [
    global_agr,
    strata_agr.get("positive", 0),
    strata_agr.get("negative", 0),
    strata_agr.get("neutral",  0),
]
colors = ["#4e79a7", "#76b7b2", "#f28e2b", "#bab0ac"]
bars = axes[0].bar(bar_labels, bar_vals, color=colors)
axes[0].set_ylim(0, 1.12)
axes[0].axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
axes[0].set_ylabel("Agreement rate")
axes[0].set_title(f"E.3 Correlation class agreement\n{GENERATOR_LABEL}")
for bar, v in zip(bars, bar_vals):
    axes[0].text(bar.get_x() + bar.get_width() / 2, v + 0.01,
                 f"{v:.3f}", ha="center", va="bottom", fontsize=9)

axes[1].bar(["False-correlation rate\n(neutral \u2192 correlated)"], [false_corr], color="#e15759")
axes[1].set_ylim(0, max(false_corr * 1.3, 0.1))
axes[1].set_ylabel("Rate")
axes[1].set_title(f"E.3 False-correlation rate\n{GENERATOR_LABEL}")
axes[1].text(0, false_corr + 0.002, f"{false_corr:.3f}",
             ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Chunk 12 — E.4: Discretized joint distributions — bivariate accuracy
#
# For each feature pair (i, j):
#   1. Fit equal-frequency bins (<=MAX_BINS) on real data for each feature.
#   2. Apply the same discretization to synthetic data (clipped to real range).
#   3. Build normalized 2D joint contingency tables.
#   4. acc = 1 - TV_distance = 1 - 0.5 * sum(|P_real - P_syn|)  in [0, 1]
#      where 1 = perfect agreement between joint distributions.
#
# Pairs are then stratified by the real-data Pearson stratum from E.3.
# Reference: https://arxiv.org/html/2504.01908

print(f"Discretizing {len(FEATURES)} features (max {MAX_BINS} bins each)...")

real_arr = real_x[FEATURES].to_numpy(float)
gen_arr  = gen_x[FEATURES].to_numpy(float)

# Pre-discretize every column
bin_real = np.empty_like(real_arr, dtype=np.int32)
bin_syn  = np.empty((gen_arr.shape[0], len(FEATURES)), dtype=np.int32)
n_bins_per_feat = []

for j in range(len(FEATURES)):
    col_r = real_arr[:, j]
    n_b = max(2, min(MAX_BINS, int(pd.Series(col_r).nunique())))
    disc = KBinsDiscretizer(n_bins=n_b, encode="ordinal", strategy="quantile")
    disc.fit(col_r.reshape(-1, 1))
    bin_real[:, j] = disc.transform(col_r.reshape(-1, 1)).ravel().astype(np.int32)
    col_s = np.clip(gen_arr[:, j], col_r.min(), col_r.max())
    bin_syn[:, j]  = disc.transform(col_s.reshape(-1, 1)).ravel().astype(np.int32)
    n_bins_per_feat.append(n_b)

print("Discretization done.")

# Enumerate pairs
all_pairs = list(combinations(range(len(FEATURES)), 2))
if MAX_PAIRS is not None and len(all_pairs) > MAX_PAIRS:
    rng_p = np.random.default_rng(42)
    idx_p = rng_p.choice(len(all_pairs), size=MAX_PAIRS, replace=False)
    sampled_pairs = [all_pairs[k] for k in sorted(idx_p)]
    print(f"Sampling {MAX_PAIRS:,} of {len(all_pairs):,} pairs.")
else:
    sampled_pairs = all_pairs
    print(f"Computing all {len(all_pairs):,} pairs.")


def _bivariate_acc(r_i, r_j, s_i, s_j, n_bi, n_bj) -> float:
    joint_r = np.zeros((n_bi, n_bj))
    joint_s = np.zeros((n_bi, n_bj))
    np.add.at(joint_r, (r_i, r_j), 1)
    np.add.at(joint_s, (s_i, s_j), 1)
    joint_r /= joint_r.sum()
    tot_s = joint_s.sum()
    if tot_s > 0:
        joint_s /= tot_s
    return float(1.0 - 0.5 * np.abs(joint_r - joint_s).sum())


# Build a lookup for pair stratum (from E.3 label arrays already computed)
feat_idx = {f: i for i, f in enumerate(FEATURES)}

acc_records = []
for ci, cj in sampled_pairs:
    acc = _bivariate_acc(
        bin_real[:, ci], bin_real[:, cj],
        bin_syn[:, ci],  bin_syn[:, cj],
        n_bins_per_feat[ci], n_bins_per_feat[cj],
    )
    acc_records.append({
        "feat_i":        FEATURES[ci],
        "feat_j":        FEATURES[cj],
        "corr_stratum":  _label_r(float(Cr[ci, cj])),
        "acc_bivariate": acc,
    })

bivariate_df = pd.DataFrame(acc_records)

print(f"\nE.4 Bivariate Accuracy Summary ({GENERATOR_LABEL}):")
print(f"  Overall mean   : {bivariate_df['acc_bivariate'].mean():.4f}")
print(f"  Overall median : {bivariate_df['acc_bivariate'].median():.4f}")
display(
    bivariate_df.groupby("corr_stratum")["acc_bivariate"]
    .agg(["count", "mean", "median", "std"])
    .reindex(["positive", "negative", "neutral"])
)

# Histograms: global + stratified
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

med_all = bivariate_df["acc_bivariate"].median()
axes[0].hist(bivariate_df["acc_bivariate"], bins=50, color="steelblue", edgecolor="white")
axes[0].axvline(med_all, color="red", linestyle="--", label=f"median = {med_all:.3f}")
axes[0].set_xlabel("Bivariate accuracy")
axes[0].set_ylabel("Number of pairs")
axes[0].set_title(f"E.4 Bivariate accuracy (all pairs)\n{GENERATOR_LABEL}")
axes[0].legend()

pal = {"positive": "#76b7b2", "negative": "#f28e2b", "neutral": "#bab0ac"}
for stratum, color in pal.items():
    sub = bivariate_df.loc[bivariate_df["corr_stratum"] == stratum, "acc_bivariate"]
    if len(sub) > 0:
        axes[1].hist(sub, bins=40, alpha=0.65, color=color, edgecolor="white",
                     label=f"{stratum} (n={len(sub):,})")
axes[1].set_xlabel("Bivariate accuracy")
axes[1].set_ylabel("Number of pairs")
axes[1].set_title(f"E.4 Bivariate accuracy by correlation stratum\n{GENERATOR_LABEL}")
axes[1].legend()
plt.tight_layout()
plt.show()